In [34]:
import pandas as pd 
import yaml
import sys

users_df = pd.read_csv("../data1/users.csv") 

with open("../data1/books.yaml") as books:
    data=yaml.safe_load(books) 
books_df = pd.DataFrame(data)

orders_df = pd.read_parquet("../data1/orders.parquet")

In [35]:
users_df["phone"] = users_df["phone"].str.replace(r"\D", "", regex=True)

In [36]:
users_df["phone"].head(10)

0    4623854294
1    9134664487
2    8019703335
3    8958295417
4    1137843410
5    6230904763
6    1195341919
7    8891635117
8    3601085596
9    8813289660
Name: phone, dtype: object

In [37]:
users_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3293 entries, 0 to 3292
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   id       3293 non-null   int64 
 1   name     3293 non-null   object
 2   address  3177 non-null   object
 3   phone    3293 non-null   object
 4   email    3293 non-null   object
dtypes: int64(1), object(4)
memory usage: 128.8+ KB


In [38]:
import numpy as np
users_df['address'] = users_df['address'].replace(r'^\s*$', np.nan, regex=True)

In [39]:
pd.DataFrame({
    'empty_string': (users_df == '').sum(),
    'NaN': users_df.isnull().sum(),
    'total_missing': (users_df == '').sum() + users_df.isnull().sum()
})

,empty_string,NaN,total_missing
id,0,0,0
name,0,0,0
address,0,166,166
phone,0,0,0
email,0,0,0


In [40]:
users_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3293 entries, 0 to 3292
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   id       3293 non-null   int64 
 1   name     3293 non-null   object
 2   address  3127 non-null   object
 3   phone    3293 non-null   object
 4   email    3293 non-null   object
dtypes: int64(1), object(4)
memory usage: 128.8+ KB


In [41]:
books_df.columns = books_df.columns.str.strip(':')
books_df.columns

Index(['id', 'title', 'author', 'genre', 'publisher', 'year'], dtype='object')

In [42]:
books_df['publisher'] = books_df['publisher'].replace(r'^\s*$', np.nan, regex=True)

In [43]:
books_df['year'] = books_df['year'].replace(r'^\s*$', np.nan, regex=True)

In [44]:
pd.DataFrame({
    'empty_string': (books_df == '').sum(),
    'NaN': books_df.isnull().sum(),
    'total_missing': (books_df == '').sum() + books_df.isnull().sum()
})

,empty_string,NaN,total_missing
id,0,0,0
title,0,0,0
author,0,0,0
genre,0,0,0
publisher,0,10,10
year,0,6,6


In [45]:
import re

def extract_date(ts):
    if pd.isnull(ts):
        return None
    ts = str(ts)
    patterns = [
        r'\d{4}-\d{2}-\d{2}',                    # 2025-07-02
        r'\d{2}/\d{2}/\d{2,4}',                  # 10/01/24, 02/07/25
        r'\d{1,2}-[A-Za-z]+-\d{4}',              # 28-August-2024, 11-November-2024
        r'\d{1,2}-[A-Za-z]{3}-\d{4}',            # 19-Oct-2024, 30-Oct-2024
        r'\d{1,2}-[A-Za-z]{3}-\d{2}\b',          # 15-Jul-2024
    ]
    for pattern in patterns:
        match = re.search(pattern, ts)
        if match:
            return match.group()
    return None

In [46]:
orders_df['date'] = orders_df['timestamp'].apply(extract_date)
orders_df['date'] = pd.to_datetime(orders_df['date'], errors='coerce')

C:\Users\User\AppData\Local\Temp\ipykernel_18688\743183744.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orders_df['date'] = pd.to_datetime(orders_df['date'], errors='coerce')


In [47]:
print(orders_df['date'].isnull().sum())

397


In [48]:
mask = orders_df['date'].isnull()
orders_df[mask]['timestamp'].head(20)

83          26.11.2024 14:33:33
90          24.09.2024 21:31:31
96     Mon Oct 21 11:52:31 2024
102         05.01.2025 14:40:40
150    Thu Mar 28 13:47:42 2024
154         23.08.2024 14:24:24
155         17.01.2025 20:58:58
156          4.07.2025  0:59:59
191         25.08.2024 17:14:14
222         30.11.2024 17:54:54
318         29.01.2025 17:10:10
363          5.08.2024  5:20:20
415         21.01.2025 10:58:58
438         20.04.2024  8:01:01
443    Sun Jan 12 08:04:00 2025
534         10.01.2025  8:22:22
572         01.12.2024 08:47:47
587         02.11.2024 16:37:37
630          2.05.2025  9:11:11
676         20.12.2024 20:29:29
Name: timestamp, dtype: object

In [49]:
def extract_date(ts):
    if pd.isnull(ts):
        return None
    ts = str(ts)
    patterns = [
    r'\d{4}-\d{2}-\d{2}',                   
    r'\d{2}/\d{2}/\d{2,4}',              
    r'\d{1,2}-[A-Za-z]+-\d{4}',              
    r'\d{1,2}-[A-Za-z]{3}-\d{4}',           
    r'\d{1,2}\.\d{1,2}\.\d{4}',             
    r'[A-Za-z]{3}\s[A-Za-z]{3}\s+\d{1,2}\s[\d:]+\s\d{4}',  
    ]
    for pattern in patterns:
        match = re.search(pattern, ts)
        if match:
            return match.group()
    return None

In [50]:
orders_df['date'] = orders_df['timestamp'].apply(extract_date)
orders_df['date'] = pd.to_datetime(orders_df['date'], errors='coerce')

C:\Users\User\AppData\Local\Temp\ipykernel_18688\743183744.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orders_df['date'] = pd.to_datetime(orders_df['date'], errors='coerce')


In [51]:
print(orders_df['date'].isnull().sum())

0


In [52]:
orders_df.head()

,id,user_id,book_id,quantity,unit_price,timestamp,shipping,date
0,71389,47288,18976,2,27.00$,10/01/24 10:38:08 A.M.,None,2024-10-01
1,66343,47049,19403,1,€50¢50,10:14;19-Oct-2024,"4940 Arnoldo Keys, West Arnette, KS 77599",2024-10-19
2,72606,46685,19500,1,USD 45.99,"22:13:35,2025-07-02",,2025-07-02
3,68462,45336,18992,1,€ 71.00,2025-10-20 16:25:20,,2025-10-20
4,72691,45311,19388,1,52.25 $,"08:48:47 A.M.,28-August-2024",None,2024-08-28


In [53]:
import re

def extract_price(val):
    if pd.isnull(val):
        return None, None
    val = str(val)
    if '$' in val or 'USD' in val:
        currency = 'USD'
    elif '€' in val or 'EUR' in val:
        currency = 'EUR'
    else:
        currency = 'UNKNOWN'
    cents_match = re.search(r'(\d+)\$(\d+)¢', val)
    if cents_match:
        amount = float(cents_match.group(1)) + float(cents_match.group(2)) / 100
        return amount, 'USD'
    match = re.search(r'[\d]+\.?\d*', val)
    if match:
        return float(match.group()), currency
    return None, currency

orders_df[['price_clean', 'currency']] = orders_df['unit_price'].apply(
    lambda x: pd.Series(extract_price(x))
)

In [54]:
print(orders_df[['unit_price', 'price_clean', 'currency']].head(20))
print(orders_df['currency'].value_counts())

   unit_price  price_clean currency
0      27.00$        27.00      USD
1      €50¢50        50.00      EUR
2   USD 45.99        45.99      USD
3     € 71.00        71.00      EUR
4     52.25 $        52.25      USD
5      22$75¢        22.75      USD
6      35 USD        35.00      USD
7   33.00 USD        33.00      USD
8    44.75USD        44.75      USD
9     49.0USD        49.00      USD
10     €46¢00        46.00      EUR
11     € 59.5        59.50      EUR
12  40.00 USD        40.00      USD
13       40 $        40.00      USD
14    USD 69.        69.00      USD
15  EUR 31.99        31.99      EUR
16   USD65.50        65.50      USD
17    55.50 $        55.50      USD
18   30.25USD        30.25      USD
19     €32¢00        32.00      EUR
currency
USD    7705
EUR    3532
Name: count, dtype: int64


In [55]:
mask = orders_df['price_clean'].isnull() | (orders_df['currency'] == 'UNKNOWN')
orders_df[mask]['unit_price'].head(20)

Series([], Name: unit_price, dtype: object)

In [56]:
books_df['year'] = pd.to_numeric(books_df['year'], errors='coerce').astype('Int64')


In [57]:
rate = 1.2

orders_df['price_usd'] = orders_df.apply(
    lambda row: round(row['price_clean'] * rate, 2)
    if row['currency'] == 'EUR'
    else round(row['price_clean'], 2),
    axis=1
)

In [58]:
orders_df['paid_price'] = orders_df['quantity'] * orders_df['price_usd']
orders_df[['quantity','unit_price', 'price_clean', 'currency', 'price_usd','paid_price']].head(10)

,quantity,unit_price,price_clean,currency,price_usd,paid_price
0,2,27.00$,27.00,USD,27.00,54.00
1,1,€50¢50,50.00,EUR,60.00,60.00
2,1,USD 45.99,45.99,USD,45.99,45.99
3,1,€ 71.00,71.00,EUR,85.20,85.20
4,1,52.25 $,52.25,USD,52.25,52.25
5,3,22$75¢,22.75,USD,22.75,68.25
6,1,35 USD,35.00,USD,35.00,35.00
7,1,33.00 USD,33.00,USD,33.00,33.00
8,1,44.75USD,44.75,USD,44.75,44.75
9,1,49.0USD,49.00,USD,49.00,49.00


In [59]:
orders_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11237 entries, 0 to 11236
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   id           11237 non-null  int64         
 1   user_id      11237 non-null  int64         
 2   book_id      11237 non-null  int64         
 3   quantity     11237 non-null  int32         
 4   unit_price   11237 non-null  object        
 5   timestamp    11237 non-null  object        
 6   shipping     8427 non-null   object        
 7   date         11237 non-null  datetime64[ns]
 8   price_clean  11237 non-null  float64       
 9   currency     11237 non-null  object        
 10  price_usd    11237 non-null  float64       
 11  paid_price   11237 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int32(1), int64(3), object(4)
memory usage: 1009.7+ KB


In [60]:
orders_df['shipping'] = orders_df['shipping'].replace(r'^\s*$', np.nan, regex=True)


In [61]:
orders_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11237 entries, 0 to 11236
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   id           11237 non-null  int64         
 1   user_id      11237 non-null  int64         
 2   book_id      11237 non-null  int64         
 3   quantity     11237 non-null  int32         
 4   unit_price   11237 non-null  object        
 5   timestamp    11237 non-null  object        
 6   shipping     5597 non-null   object        
 7   date         11237 non-null  datetime64[ns]
 8   price_clean  11237 non-null  float64       
 9   currency     11237 non-null  object        
 10  price_usd    11237 non-null  float64       
 11  paid_price   11237 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int32(1), int64(3), object(4)
memory usage: 1009.7+ KB


In [62]:
books_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 753 entries, 0 to 752
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         753 non-null    int64 
 1   title      753 non-null    object
 2   author     753 non-null    object
 3   genre      753 non-null    object
 4   publisher  743 non-null    object
 5   year       744 non-null    Int64 
dtypes: Int64(1), int64(1), object(4)
memory usage: 36.2+ KB


In [68]:
orders_df = orders_df.drop(columns=['unit_price','timestamp'])

In [69]:
orders_df.head()

,id,user_id,book_id,quantity,shipping,date,price_clean,currency,price_usd,paid_price
0,71389,47288,18976,2,None,2024-10-01,27.00,USD,27.00,54.00
1,66343,47049,19403,1,"4940 Arnoldo Keys, West Arnette, KS 77599",2024-10-19,50.00,EUR,60.00,60.00
2,72606,46685,19500,1,NaN,2025-07-02,45.99,USD,45.99,45.99
3,68462,45336,18992,1,NaN,2025-10-20,71.00,EUR,85.20,85.20
4,72691,45311,19388,1,None,2024-08-28,52.25,USD,52.25,52.25


In [70]:
books_df.head()

,id,title,author,genre,publisher,year
0,19199,The Yellow Meads of Asphodel,Carolyne West,Classic,Mainstream Publishing,2009
1,19398,From Here to Eternity,"Rep. Heath Stiedemann, Gino Welch, Haydee Larson",Short story,Vintage Books,2001
2,19483,Eyeless in Gaza,Vannessa Price,Biography/Autobiography,Pavilion Books,1886
3,19506,Precious Bane,Miss Yong Wyman,Realistic fiction,New English Library,2021
4,19570,City of God,Travis Moore,Suspense/Thriller,Bellevue Literary Press,1847


In [71]:
users_df.head()

,id,name,address,phone,email
0,44533,Hoyt Carter,"Apt. 300 8604 Ashlyn Wells, Effertzstad, ID 02997",4623854294,mckinley.rowe@harber.example
1,46128,Marco Kulas,"Apt. 538 816 Bechtelar Ferry, Lincolnhaven, KS...",9134664487,francisco@murray-cronin.test
2,46407,Denny Goyette LLD,"Apt. 174 39450 Mohr Rapids, Port Neomistad, AL...",8019703335,marguerita@wuckert.test
3,44602,Zackary Heller,"Apt. 608 74228 Bogan Valley, South Stepanieshi...",8958295417,annabelle@hessel.test
4,45828,Jess Beier,"2716 Jacobi Path, Ziemanntown, SC 65624-4660",1137843410,minh@hettinger.example


In [74]:
books_df.to_csv('../data1/books_clean.csv', index=False)
users_df.to_csv('../data1/users_clean.csv', index=False)
orders_df.to_csv('../data1/orders_clean.csv', index=False)